In [ ]:
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForMaskedLM, get_linear_schedule_with_warmup
from torch.optim import AdamW
from torch.cuda.amp import GradScaler, autocast
import os
import pandas as pd
import json

# ========================
# 1. Model & Tokenizer Initialization
# ========================
tokenizer = AutoTokenizer.from_pretrained("impyadav/GPT2-FineTuned-Hinglish-Song-Generation")
model = AutoModelForCausalLM.from_pretrained("impyadav/GPT2-FineTuned-Hinglish-Song-Generation")

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.unk_token

special_tokens = [
    "[Sentiment]", "[Rhyme]", "[Tempo]", "[Energy]", "[Loudness]",
    "[Danceability]", "[Speechiness]", "[Acousticness]", "[Instrumentalness]",
    "[Liveness]", "[Valence]", "[Explicit]", "[Popularity]", "[Chroma]",
    "[SpectralContrast]", "[ZeroCrossings]", "[RhymeScheme]"
]
tokenizer.add_special_tokens({'additional_special_tokens': special_tokens})
model.resize_token_embeddings(len(tokenizer))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# ========================
# 2. Prepare the Dataset
# ========================

file_path = "/kaggle/input/hindi-songs/hindi_songs_with_sentiment.json"
with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)
df = pd.DataFrame(data)

class HinglishLyricsDataset(Dataset):
    def __init__(self, df, tokenizer, max_seq_len=512):
        self.df = df
        self.tokenizer = tokenizer
        self.max_seq_len = max_seq_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        lyrics = row.get("Hindi Lyrics", "")
        sentiment = row.get("Sentiment", "NA")
        rhyme = row.get("rhyme_dict", "NA")
        tempo = str(row.get("tempo", "NA"))
        energy = str(row.get("energy", "NA"))
        loudness = str(row.get("loudness", "NA"))
        danceability = str(row.get("danceability", "NA"))
        speechiness = str(row.get("speechiness", "NA"))
        acousticness = str(row.get("acousticness", "NA"))
        instrumentalness = str(row.get("instrumentalness", "NA"))
        liveness = str(row.get("liveness", "NA"))
        valence = str(row.get("valence", "NA"))
        explicit = str(row.get("explicit", "NA"))
        popularity = str(row.get("popularity", "NA"))
        chroma = str(row.get("chroma", "NA"))
        spectral_contrast = str(row.get("spectral_contrast", "NA"))
        zero_crossings = str(row.get("zero_crossings", "NA"))
        rhyme_scheme = row.get("rhyme_scheme", "NA")
        
        prompt = (
            f"[Sentiment]: {sentiment} {tokenizer.eos_token} "
            f"[Rhyme]: {rhyme} {tokenizer.eos_token} "
            f"[Tempo]: {tempo} {tokenizer.eos_token} "
            f"[Energy]: {energy} {tokenizer.eos_token} "
            f"[Loudness]: {loudness} {tokenizer.eos_token} "
            f"[Danceability]: {danceability} {tokenizer.eos_token} "
            f"[Speechiness]: {speechiness} {tokenizer.eos_token} "
            f"[Acousticness]: {acousticness} {tokenizer.eos_token} "
            f"[Instrumentalness]: {instrumentalness} {tokenizer.eos_token} "
            f"[Liveness]: {liveness} {tokenizer.eos_token} "
            f"[Valence]: {valence} {tokenizer.eos_token} "
            f"[Explicit]: {explicit} {tokenizer.eos_token} "
            f"[Popularity]: {popularity} {tokenizer.eos_token} "
            f"[Chroma]: {chroma} {tokenizer.eos_token} "
            f"[SpectralContrast]: {spectral_contrast} {tokenizer.eos_token} "
            f"[ZeroCrossings]: {zero_crossings} {tokenizer.eos_token} "
            f"[RhymeScheme]: {rhyme_scheme}\n"
        )
        
        combined_text = prompt + lyrics
        combined_enc = self.tokenizer(
            combined_text,
            return_tensors='pt',
            padding="max_length",
            truncation=True,
            max_length=self.max_seq_len
        )
        
        input_ids = combined_enc.input_ids.squeeze()
        attention_mask = combined_enc.attention_mask.squeeze()

        prompt_len = len(self.tokenizer.encode(prompt, add_special_tokens=False))
        labels = input_ids.clone()
        labels[:prompt_len] = -100

        return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

max_seq_len = 512
batch_size = 4
dataset_obj = HinglishLyricsDataset(df, tokenizer, max_seq_len=max_seq_len)
dataloader = DataLoader(dataset_obj, batch_size=batch_size, shuffle=True)

# ========================
# 3. Training Setup
# ========================

optimizer = AdamW(model.parameters(), lr=5e-5)
scaler = GradScaler()
epochs = 5
total_steps = len(dataloader) * epochs
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=total_steps // 10, num_training_steps=total_steps
)

accumulation_steps = 8

# ========================
# 4. Training Loop
# ========================

model.train()
for epoch in range(epochs):
    running_loss = 0.0
    for i, batch in enumerate(dataloader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        with autocast():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss.mean() / accumulation_steps

        scaler.scale(loss).backward()
        if (i + 1) % accumulation_steps == 0 or (i + 1) == len(dataloader):
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step()

        running_loss += loss.item() * accumulation_steps
        if (i + 1) % 50 == 0:
            avg_loss = running_loss / 50
            print(f"Epoch [{epoch+1}/{epochs}], Step [{i+1}/{len(dataloader)}], Loss: {avg_loss:.4f}")
            running_loss = 0.0
    torch.cuda.empty_cache()

print("Training complete.")

# ========================
# 5. Save the Fine-Tuned Model & Tokenizer
# ========================

save_path = "./fine_tuned_impyadav_MURIL"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

# ========================
# 6. Generation Function
# ========================

def generate_lyrics(seed, sentiment, rhyme, tempo, max_len=150):
    model.eval()
    prompt = (
        f"[Sentiment]: {sentiment} {tokenizer.eos_token} "
        f"[Rhyme]: {rhyme} {tokenizer.eos_token} "
        f"[Tempo]: {tempo}\n"
        f"{seed}"
    )
    
    input_ids = tokenizer.encode(prompt, return_tensors='pt', truncation=True, max_length=512).to(device)
    
    with torch.no_grad():
        outputs = model.generate(
            input_ids,
            max_length=max_len,
            num_return_sequences=1,
            temperature=1.0,
            top_p=0.95,
            repetition_penalty=1.2,
            do_sample=True
        )
    
    lyrics = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return lyrics

# ========================
# 7. Example Generation
# ========================

generated = generate_lyrics("Tera chehra", "Romantic", "AABB", "120")
print("Generated Lyrics:\n", generated)


In [14]:
def generate_lyrics(seed, sentiment, rhyme, tempo, max_len=150):
    model.eval()
    prompt = (
        f"[Sentiment]: {sentiment} {tokenizer.eos_token} "
        f"[Rhyme]: {rhyme} {tokenizer.eos_token} "
        f"[Tempo]: {tempo}\n"
        f"{seed}"
    )
    
    input_ids = tokenizer.encode(prompt, return_tensors='pt', truncation=True, max_length=512).to(device)
    
    with torch.no_grad():
        outputs = model.generate(
            input_ids,
            max_length=max_len,
            num_return_sequences=1,
            temperature=1.0,
            top_p=0.95,
            repetition_penalty=1.2,
            do_sample=True
        )
    
    lyrics = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return lyrics

# ========================
# 7. Example Generation
# ========================

generated = generate_lyrics("Pal pal jeena muhaal mera tere bina", "Satisfaction", "AABB", "120")
print("Generated Lyrics:\n", generated)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Generated Lyrics:
 : Satisfaction  : AABB  : 120
Pal pal jeena muhaal mera tere bina hai dil ko baandhan ho gaya Kya hua main, kya hua deewana yaar apno ko chahat hi badh pe marz ki aangan ho gaye  Maasoom mein yehi saawan humse sanam jaise bas yahan honthon se har din pyar ke sapne doge tujhe uske raste na dillagi ka phisalmaan ho gayy - (2)  Raaton ne woh jab raatein suno mujhe todh nahi


In [2]:
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, get_linear_schedule_with_warmup
from torch.optim import AdamW
from torch.cuda.amp import GradScaler, autocast
import os
import pandas as pd
import json
import re

# ========================
# 1. Model & Tokenizer Initialization
# ========================
tokenizer = AutoTokenizer.from_pretrained("impyadav/GPT2-FineTuned-Hinglish-Song-Generation")
model = AutoModelForCausalLM.from_pretrained("impyadav/GPT2-FineTuned-Hinglish-Song-Generation")

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.unk_token

special_tokens = [
    "[Sentiment]", "[Rhyme]", "[Tempo]", "[Energy]", "[Loudness]",
    "[Danceability]", "[Speechiness]", "[Acousticness]", "[Instrumentalness]",
    "[Liveness]", "[Valence]", "[Explicit]", "[Popularity]", "[Chroma]",
    "[SpectralContrast]", "[ZeroCrossings]", "[RhymeScheme]", "[Stanza]", 
    "[EndStanza]", "[Line]", "[EndLine]", "[A]", "[B]", "[C]", "[D]", "[E]", 
    "[F]", "[G]", "[H]", "[I]", "[J]", "[K]", "[L]", "[M]", "[N]", "[O]", "[P]"
]
tokenizer.add_special_tokens({'additional_special_tokens': special_tokens})
model.resize_token_embeddings(len(tokenizer))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# ========================
# 2. Prepare the Dataset
# ========================

file_path = "/kaggle/input/hindi-songs/hindi_songs_with_sentiment.json"
with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)
df = pd.DataFrame(data)

class HinglishLyricsDataset(Dataset):
    def __init__(self, df, tokenizer, max_seq_len=512):
        self.df = df
        self.tokenizer = tokenizer
        self.max_seq_len = max_seq_len

    def __len__(self):
        return len(self.df)
    
    def parse_lyrics_with_rhyme(self, lyrics_str, rhyme_scheme, rhyme_dict):
        # Check if lyrics is a string representation of a list
        if isinstance(lyrics_str, str) and lyrics_str.startswith('[') and lyrics_str.endswith(']'):
            try:
                # Remove the 'Lyrics' entry and any numeric entry at the end if present
                lyrics_list = eval(lyrics_str)
                if lyrics_list and lyrics_list[0] == 'Lyrics':
                    lyrics_list = lyrics_list[1:]
                # Remove any numeric entries at the end
                while lyrics_list and lyrics_list[-1].isdigit():
                    lyrics_list.pop()
            except:
                # If eval fails, split by commas and clean up
                lyrics_list = lyrics_str.strip('[]').split(',')
                lyrics_list = [line.strip().strip("'\"") for line in lyrics_list]
        else:
            # If it's already a clean string, just split by newlines
            lyrics_list = lyrics_str.split('\n')
        
        # Clean up empty lines and prepare for stanza detection
        cleaned_lyrics = []
        for line in lyrics_list:
            line = line.strip().strip("'\"")
            if line:  # Skip empty lines for structured format
                cleaned_lyrics.append(line)
        
        # Parse rhyme scheme to match with lines
        rhyme_labels = []
        if isinstance(rhyme_scheme, str) and rhyme_scheme.startswith('[') and rhyme_scheme.endswith(']'):
            try:
                rhyme_labels = eval(rhyme_scheme)
            except:
                rhyme_labels = rhyme_scheme.strip('[]').split(',')
                rhyme_labels = [label.strip().strip("'\"") for label in rhyme_labels]
        
        # Ensure we have the same number of rhyme labels as lyrics lines
        if len(rhyme_labels) != len(cleaned_lyrics):
            # Pad or truncate rhyme labels to match lyrics length
            if len(rhyme_labels) < len(cleaned_lyrics):
                rhyme_labels.extend(['X'] * (len(cleaned_lyrics) - len(rhyme_labels)))
            else:
                rhyme_labels = rhyme_labels[:len(cleaned_lyrics)]
        
        # Detect stanzas by looking for consecutive empty lines in original list
        stanza_breaks = []
        stanza_count = 0
        in_stanza = False
        
        for i, line in enumerate(lyrics_list):
            line = line.strip().strip("'\"")
            if line and not in_stanza:
                in_stanza = True
                stanza_count += 1
            elif not line and in_stanza:
                in_stanza = False
                stanza_breaks.append(i)
        
        # Format lyrics with stanza and rhyme information
        formatted_lyrics = "[Stanza]1[EndStanza]\n"
        current_stanza = 1
        stanza_line_count = 0
        
        for i, (line, rhyme) in enumerate(zip(cleaned_lyrics, rhyme_labels)):
            # Check if we need to start a new stanza
            if stanza_breaks and i >= stanza_breaks[0] and stanza_line_count > 0:
                stanza_breaks.pop(0)
                current_stanza += 1
                formatted_lyrics += f"\n[Stanza]{current_stanza}[EndStanza]\n"
                stanza_line_count = 0
            
            # Add the line with rhyme tag
            formatted_lyrics += f"[Line]{line}[{rhyme}][EndLine]\n"
            stanza_line_count += 1
        
        return formatted_lyrics, rhyme_dict

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        lyrics = row.get("Hindi Lyrics", "")
        sentiment = row.get("Sentiment", "NA")
        rhyme_dict = row.get("rhyme_dict", "{}")
        rhyme_scheme = row.get("rhyme_scheme", "[]")
        tempo = str(row.get("tempo", "NA"))
        energy = str(row.get("energy", "NA"))
        loudness = str(row.get("loudness", "NA"))
        danceability = str(row.get("danceability", "NA"))
        speechiness = str(row.get("speechiness", "NA"))
        acousticness = str(row.get("acousticness", "NA"))
        instrumentalness = str(row.get("instrumentalness", "NA"))
        liveness = str(row.get("liveness", "NA"))
        valence = str(row.get("valence", "NA"))
        explicit = str(row.get("explicit", "NA"))
        popularity = str(row.get("popularity", "NA"))
        chroma = str(row.get("chroma", "NA"))
        spectral_contrast = str(row.get("spectral_contrast", "NA"))
        zero_crossings = str(row.get("zero_crossings", "NA"))
        
        # Parse and format lyrics with rhyme scheme
        formatted_lyrics, rhyme_dict = self.parse_lyrics_with_rhyme(lyrics, rhyme_scheme, rhyme_dict)
        
        prompt = (
            f"[Sentiment]: {sentiment} {tokenizer.eos_token} "
            f"[Rhyme]: {rhyme_dict} {tokenizer.eos_token} "
            f"[Tempo]: {tempo} {tokenizer.eos_token} "
            f"[Energy]: {energy} {tokenizer.eos_token} "
            f"[Loudness]: {loudness} {tokenizer.eos_token} "
            f"[Danceability]: {danceability} {tokenizer.eos_token} "
            f"[Speechiness]: {speechiness} {tokenizer.eos_token} "
            f"[Acousticness]: {acousticness} {tokenizer.eos_token} "
            f"[Instrumentalness]: {instrumentalness} {tokenizer.eos_token} "
            f"[Liveness]: {liveness} {tokenizer.eos_token} "
            f"[Valence]: {valence} {tokenizer.eos_token} "
            f"[Explicit]: {explicit} {tokenizer.eos_token} "
            f"[Popularity]: {popularity} {tokenizer.eos_token} "
            f"[Chroma]: {chroma} {tokenizer.eos_token} "
            f"[SpectralContrast]: {spectral_contrast} {tokenizer.eos_token} "
            f"[ZeroCrossings]: {zero_crossings} {tokenizer.eos_token} "
            f"[RhymeScheme]: {rhyme_scheme}\n"
        )
        
        combined_text = prompt + formatted_lyrics
        combined_enc = self.tokenizer(
            combined_text,
            return_tensors='pt',
            padding="max_length",
            truncation=True,
            max_length=self.max_seq_len
        )
        
        input_ids = combined_enc.input_ids.squeeze()
        attention_mask = combined_enc.attention_mask.squeeze()

        prompt_len = len(self.tokenizer.encode(prompt, add_special_tokens=False))
        labels = input_ids.clone()
        labels[:prompt_len] = -100

        return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

max_seq_len = 512
batch_size = 4
dataset_obj = HinglishLyricsDataset(df, tokenizer, max_seq_len=max_seq_len)
dataloader = DataLoader(dataset_obj, batch_size=batch_size, shuffle=True)

# ========================
# 3. Training Setup
# ========================

optimizer = AdamW(model.parameters(), lr=5e-5)
scaler = GradScaler()
epochs = 5
total_steps = len(dataloader) * epochs
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=total_steps // 10, num_training_steps=total_steps
)

accumulation_steps = 8

# ========================
# 4. Training Loop
# ========================

model.train()
for epoch in range(epochs):
    running_loss = 0.0
    for i, batch in enumerate(dataloader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        with autocast():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss.mean() / accumulation_steps

        scaler.scale(loss).backward()
        if (i + 1) % accumulation_steps == 0 or (i + 1) == len(dataloader):
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step()

        running_loss += loss.item() * accumulation_steps
        if (i + 1) % 50 == 0:
            avg_loss = running_loss / 50
            print(f"Epoch [{epoch+1}/{epochs}], Step [{i+1}/{len(dataloader)}], Loss: {avg_loss:.4f}")
            running_loss = 0.0
    
    # Save checkpoint at the end of each epoch
    save_checkpoint_path = f"./checkpoints/epoch_{epoch+1}"
    os.makedirs(save_checkpoint_path, exist_ok=True)
    model.save_pretrained(save_checkpoint_path)
    tokenizer.save_pretrained(save_checkpoint_path)
    print(f"Checkpoint saved at epoch {epoch+1}")
    
    torch.cuda.empty_cache()

print("Training complete.")

# ========================
# 5. Save the Fine-Tuned Model & Tokenizer
# ========================

save_path = "./fine_tuned_impyadav_GPT2_hindi_lyrics"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

# ========================
# 6. Generate Rhyming Dictionary
# ========================

def generate_rhyme_dict(rhyme_pattern="AABB"):
    """Create a sample rhyme dictionary based on common Hindi rhyming sounds"""
    common_hindi_endings = {
        'A': 'na',
        'B': 'ye',
        'C': 'ta',
        'D': 'aa',
        'E': 'ra',
        'F': 'di',
        'G': 'ki',
        'H': 'se',
        'I': 'le',
        'J': 'ho',
        'K': 'ka',
        'L': 'ja',
        'M': 'ni',
        'N': 'me',
        'O': 'ga',
        'P': 'sa'
    }
    
    # Extract unique letters from the rhyme pattern
    unique_letters = ''.join(set(rhyme_pattern))
    
    # Create dictionary with only the required endings
    rhyme_dict = {}
    for letter in unique_letters:
        if letter in common_hindi_endings:
            rhyme_dict[letter] = common_hindi_endings[letter]
    
    return str(rhyme_dict)

# ========================
# 7. Enhanced Generation Function
# ========================

def generate_lyrics(
    seed, 
    sentiment, 
    rhyme_pattern="AABB", 
    tempo=120, 
    energy=0.5,
    loudness=-30,
    danceability=0.5,
    num_stanzas=2,
    lines_per_stanza=4,
    max_len=512
):
    model.eval()
    
    # Generate rhyme dictionary based on the pattern
    rhyme_dict = generate_rhyme_dict(rhyme_pattern)
    
    prompt = (
        f"[Sentiment]: {sentiment} {tokenizer.eos_token} "
        f"[Rhyme]: {rhyme_dict} {tokenizer.eos_token} "
        f"[Tempo]: {tempo} {tokenizer.eos_token} "
        f"[Energy]: {energy} {tokenizer.eos_token} "
        f"[Loudness]: {loudness} {tokenizer.eos_token} "
        f"[Danceability]: {danceability} {tokenizer.eos_token} "
        f"[RhymeScheme]: {rhyme_pattern}\n"
        f"[Stanza]1[EndStanza]\n"
        f"{seed}"
    )
    
    input_ids = tokenizer.encode(prompt, return_tensors='pt', truncation=True, max_length=512).to(device)
    
    # Define stopping criteria to avoid generating beyond a reasonable length
    stop_token_ids = [tokenizer.eos_token_id]
    
    with torch.no_grad():
        outputs = model.generate(
            input_ids,
            max_length=max_len,
            num_return_sequences=1,
            temperature=1.0,
            top_p=0.95,
            repetition_penalty=1.2,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            no_repeat_ngram_size=3
        )
    
    raw_lyrics = tokenizer.decode(outputs[0], skip_special_tokens=False)
    
    # Post-process to format the lyrics nicely
    formatted_lyrics = post_process_lyrics(raw_lyrics)
    
    return formatted_lyrics

def post_process_lyrics(raw_lyrics):
    """
    Post-process the raw generated lyrics to format them nicely with stanzas and rhyme schemes
    """
    # Extract only the generated lyrics part (after the prompt)
    lines = raw_lyrics.split('\n')
    start_idx = 0
    for i, line in enumerate(lines):
        if "[Stanza]1[EndStanza]" in line:
            start_idx = i
            break
    
    # Extract lyrics content
    lyrics_content = lines[start_idx:]
    
    # Clean up formatting
    formatted_output = ""
    current_stanza = 0
    
    for line in lyrics_content:
        # Handle stanza markers
        if "[Stanza]" in line:
            stanza_match = re.search(r'\[Stanza\](\d+)\[EndStanza\]', line)
            if stanza_match:
                stanza_num = stanza_match.group(1)
                if current_stanza > 0:  # Add extra newline between stanzas
                    formatted_output += "\n"
                formatted_output += f"Stanza_{stanza_num}:\n"
                current_stanza = int(stanza_num)
        
        # Handle line content with rhyme scheme
        elif "[Line]" in line:
            # Extract line content and rhyme tag
            line_match = re.search(r'\[Line\](.*?)\[(.*?)\]\[EndLine\]', line)
            if line_match:
                line_content = line_match.group(1)
                rhyme_tag = line_match.group(2)
                formatted_output += f"{line_content} ({rhyme_tag})\n"
        
        # Handle any other line that might have been generated without proper tags
        elif line.strip() and not any(tag in line for tag in ["[Sentiment]", "[Rhyme]", "[Tempo]"]):
            formatted_output += f"{line}\n"
    
    return formatted_output

# ========================
# 8. Example Generation
# ========================

# Example 1: Romantic song with AABB rhyme pattern
romantic_lyrics = generate_lyrics(
    seed="Tere bina", 
    sentiment="Romantic, love, passion", 
    rhyme_pattern="AABB", 
    tempo=90,
    energy=0.4,
    danceability=0.6
)
print("Generated Romantic Lyrics:\n", romantic_lyrics)

# Example 2: Sad song with ABAB rhyme pattern
sad_lyrics = generate_lyrics(
    seed="Dil ke", 
    sentiment="Sad, melancholy, heartbreak", 
    rhyme_pattern="ABAB", 
    tempo=70,
    energy=0.3,
    danceability=0.4
)

print("\nGenerated Sad Lyrics:\n", sad_lyrics)

# Example 3: Party song with AAAA rhyme pattern
party_lyrics = generate_lyrics(
    seed="Aaj raat", 
    sentiment="Happy, energetic, celebration", 
    rhyme_pattern="AAAA", 
    tempo=130,
    energy=0.8,
    danceability=0.9
)
print("\nGenerated Party Lyrics:\n", party_lyrics)

pytorch_model.bin:  21%|##1       | 304M/1.44G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.44G [00:00<?, ?B/s]

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
/tmp/ipykernel_31/2473694149.py:200: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipykernel_31/2473694149.py:221: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Epoch [1/5], Step [50/576], Loss: 7.1652
Epoch [1/5], Step [100/576], Loss: 6.7088
Epoch [1/5], Step [150/576], Loss: 6.1524
Epoch [1/5], Step [200/576], Loss: 5.6210
Epoch [1/5], Step [250/576], Loss: 5.0940
Epoch [1/5], Step [300/576], Loss: 4.9202
Epoch [1/5], Step [350/576], Loss: 4.9489
Epoch [1/5], Step [400/576], Loss: 4.7742
Epoch [1/5], Step [450/576], Loss: 4.6619
Epoch [1/5], Step [500/576], Loss: 4.5117
Epoch [1/5], Step [550/576], Loss: 4.4666
Checkpoint saved at epoch 1


/tmp/ipykernel_31/2473694149.py:221: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


...

Epoch [4/5], Step [50/576], Loss: 1.9692
Epoch [4/5], Step [100/576], Loss: 1.9059
Epoch [4/5], Step [150/576], Loss: 1.8260
Epoch [4/5], Step [200/576], Loss: 1.8543
Epoch [4/5], Step [250/576], Loss: 1.7988
Epoch [4/5], Step [300/576], Loss: 1.8526
Epoch [4/5], Step [350/576], Loss: 1.8454
Epoch [4/5], Step [400/576], Loss: 1.7601
Epoch [4/5], Step [450/576], Loss: 1.7264
Epoch [4/5], Step [500/576], Loss: 1.7373
Epoch [4/5], Step [550/576], Loss: 1.7344
Checkpoint saved at epoch 4


/tmp/ipykernel_31/2473694149.py:221: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch [5/5], Step [50/576], Loss: 1.6519
Epoch [5/5], Step [100/576], Loss: 1.6350
Epoch [5/5], Step [150/576], Loss: 1.6841
Epoch [5/5], Step [200/576], Loss: 1.6254
Epoch [5/5], Step [250/576], Loss: 1.6616
Epoch [5/5], Step [300/576], Loss: 1.5843
Epoch [5/5], Step [350/576], Loss: 1.5994
Epoch [5/5], Step [400/576], Loss: 1.6079
Epoch [5/5], Step [450/576], Loss: 1.5552
Epoch [5/5], Step [500/576], Loss: 1.6097
Epoch [5/5], Step [550/576], Loss: 1.5766
Checkpoint saved at epoch 5
Training complete.


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Generated Romantic Lyrics:
 Stanza_1:
Tere bina se hai pyar yeh na[C][EndLine]


Generated Sad Lyrics:
 Stanza_1:
Dil ke saath hai dar gaya[D][EndLine]


Generated Party Lyrics:
 Stanza_1:
Aaj raat karaar main ... na[C]



# attempt:1

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import re

# ========================
# 1. Model & Tokenizer Initialization
# ========================
tokenizer = AutoTokenizer.from_pretrained("/kaggle/working/fine_tuned_impyadav_GPT2_hindi_lyrics")
model = AutoModelForCausalLM.from_pretrained("/kaggle/working/fine_tuned_impyadav_GPT2_hindi_lyrics")

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Add special tokens if they don't already exist
special_tokens = [
    "[Sentiment]", "[Rhyme]", "[Tempo]", "[Energy]", "[Loudness]",
    "[Danceability]", "[Speechiness]", "[Acousticness]", "[Instrumentalness]",
    "[Liveness]", "[Valence]", "[Explicit]", "[Popularity]", "[Chroma]",
    "[SpectralContrast]", "[ZeroCrossings]", "[RhymeScheme]"
]
# Only add tokens that don't already exist
tokens_to_add = []
for token in special_tokens:
    if token not in tokenizer.get_vocab():
        tokens_to_add.append(token)

if tokens_to_add:
    tokenizer.add_special_tokens({'additional_special_tokens': tokens_to_add})
    model.resize_token_embeddings(len(tokenizer))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# ========================
# 2. Enhanced Lyrics Generation
# ========================

def generate_lyrics(
    seed, 
    sentiment, 
    rhyme_dict=None, 
    rhyme_scheme=None,
    tempo=120, 
    energy=0.5,
    loudness=-30,
    danceability=0.5,
    max_length=512
):
    model.eval()
    
    # Default rhyme scheme and dict if not provided
    if not rhyme_scheme:
        rhyme_scheme = "['A', 'A', 'B', 'B', 'C', 'C', 'D', 'D']"
    
    if not rhyme_dict:
        rhyme_dict = "{'A': 'na', 'B': 'ye', 'C': 'ra', 'D': 'di'}"
    
    # Build prompt with all required conditioning
    prompt = (
        f"[Sentiment]: {sentiment} {tokenizer.eos_token} "
        f"[Rhyme]: {rhyme_dict} {tokenizer.eos_token} "
        f"[Tempo]: {tempo} {tokenizer.eos_token} "
        f"[Energy]: {energy} {tokenizer.eos_token} "
        f"[Loudness]: {loudness} {tokenizer.eos_token} "
        f"[Danceability]: {danceability} {tokenizer.eos_token} "
        f"[RhymeScheme]: {rhyme_scheme}\n\n"
    )
    
    # Add the seed text to start generation
    if seed:
        prompt += seed
    
    # Tokenize the prompt
    inputs = tokenizer(prompt, return_tensors="pt", padding=True, truncation=True).to(device)
    
    # Generate lyrics with careful parameter tuning
    with torch.no_grad():
        generated_ids = model.generate(
            inputs.input_ids,
            attention_mask=inputs.attention_mask,
            do_sample=True,
            temperature=0.85,  # Good balance between creativity and coherence
            top_p=0.92,
            top_k=50,
            repetition_penalty=1.15,
            max_length=max_length,  # Use only max_length, not max_new_tokens
            pad_token_id=tokenizer.pad_token_id,
            num_return_sequences=1,
            no_repeat_ngram_size=2
        )
    
    # Decode the generated text
    generated_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
    
    # Clean up the generated text (remove prompt parts)
    output = clean_generated_text(generated_text, prompt)
    
    # Format with stanzas and rhyme scheme
    formatted_output = format_lyrics_with_rhyme_scheme(output, rhyme_scheme)
    
    return formatted_output

def clean_generated_text(generated_text, prompt):
    """Remove the prompt part from the generated text and clean up"""
    # Remove the prompt from the beginning
    if generated_text.startswith(prompt):
        output = generated_text[len(prompt):]
    else:
        # If exact match not found, try to find where the generation actually starts
        # by looking for the seed or the first reasonable sentence
        lines = generated_text.split('\n')
        for i, line in enumerate(lines):
            if line and not any(token in line for token in ["[Sentiment]", "[Rhyme]", "[Tempo]"]):
                output = '\n'.join(lines[i:])
                break
        else:
            output = generated_text
    
    # Remove any trailing special tokens or formatting
    output = re.sub(r'\[.*?\]', '', output).strip()
    
    return output

def format_lyrics_with_rhyme_scheme(lyrics_text, rhyme_scheme_str):
    """Format lyrics with stanza structure and rhyme labels"""
    # Parse rhyme scheme
    try:
        rhyme_scheme = eval(rhyme_scheme_str)
        if isinstance(rhyme_scheme, list):
            rhyme_pattern = rhyme_scheme
        else:
            # Default pattern if parsing fails
            rhyme_pattern = ['A', 'A', 'B', 'B', 'C', 'C', 'D', 'D']
    except:
        # Default if parsing completely fails
        rhyme_pattern = ['A', 'A', 'B', 'B', 'C', 'C', 'D', 'D']
    
    # Split lyrics into lines
    lines = [line.strip() for line in lyrics_text.split('\n') if line.strip()]
    
    # Format with stanzas (assume 4 lines per stanza) and add rhyme labels
    formatted_output = ""
    current_stanza = 1
    
    for i, line in enumerate(lines):
        # Start a new stanza
        if i % 4 == 0:
            if i > 0:
                formatted_output += "\n"  # Add blank line between stanzas
            formatted_output += f"Stanza_{current_stanza}:\n"
            current_stanza += 1
        
        # Add line with rhyme label
        rhyme_label = rhyme_pattern[i % len(rhyme_pattern)]
        formatted_output += f"{line} ({rhyme_label})\n"
    
    return formatted_output

# ========================
# 3. Examples with Real Model Generation
# ========================

# Example 1: Romantic song
print("Generating Romantic Lyrics...")
romantic_lyrics = generate_lyrics(
    seed="Tere bina", 
    sentiment="Romantic, love, passion",
    rhyme_dict="{'A': 'na', 'B': 'ye', 'C': 'di', 'D': 'ra'}",
    rhyme_scheme="['A', 'A', 'B', 'B']",
    tempo=90,
    energy=0.4,
    danceability=0.6
)
print("\nGenerated Romantic Lyrics:\n", romantic_lyrics)

# Example 2: Sad song
print("\nGenerating Sad Lyrics...")
sad_lyrics = generate_lyrics(
    seed="Dil ke dard", 
    sentiment="Sad, melancholy, heartbreak",
    rhyme_dict="{'A': 'ra', 'B': 'na', 'C': 'ya', 'D': 'ni'}",
    rhyme_scheme="['A', 'B', 'A', 'B']",
    tempo=70,
    energy=0.3,
    danceability=0.4
)
print("\nGenerated Sad Lyrics:\n", sad_lyrics)

# Example 3: Party song
print("\nGenerating Party Lyrics...")
party_lyrics = generate_lyrics(
    seed="Aaj raat ka", 
    sentiment="Happy, energetic, celebration",
    rhyme_dict="{'A': 'na', 'B': 'le', 'C': 'jo', 'D': 'ra'}",
    rhyme_scheme="['A', 'A', 'A', 'A']",
    tempo=130,
    energy=0.8,
    danceability=0.9
)
print("\nGenerated Party Lyrics:\n", party_lyrics)

# ========================
# 4. Production-Ready Function with Error Handling
# ========================

def generate_hindi_lyrics(
    seed_phrase,
    sentiment="Neutral",
    rhyme_pattern="AABB",  # Options: AABB, ABAB, AAAA, etc.
    tempo=100,
    energy=0.5,
    stanzas=2,
    lines_per_stanza=4
):
    """
    Production-ready function that handles all the complexity and provides a clean interface
    """
    try:
        # Convert rhyme pattern to rhyme scheme list format
        rhyme_scheme_list = []
        for _ in range(stanzas):
            for char in rhyme_pattern[:lines_per_stanza]:
                rhyme_scheme_list.append(char)
        
        rhyme_scheme_str = str(rhyme_scheme_list)
        
        # Common Hindi rhyming sounds based on pattern
        rhyme_dict = {
            'A': 'na',  # ends with 'na' sound
            'B': 'ye',  # ends with 'ye' sound
            'C': 'ra',  # ends with 'ra' sound
            'D': 'di',  # ends with 'di' sound
            'E': 'le',  # ends with 'le' sound
        }
        
        # Generate lyrics using the model
        lyrics = generate_lyrics(
            seed=seed_phrase,
            sentiment=sentiment,
            rhyme_dict=str(rhyme_dict),
            rhyme_scheme=rhyme_scheme_str,
            tempo=tempo,
            energy=energy
        )
        
        return lyrics
        
    except Exception as e:
        # Provide graceful error handling
        error_msg = f"Error generating lyrics: {str(e)}"
        return error_msg

# Example usage of the production function
print("\nProduction function example:")
lyrics = generate_hindi_lyrics(
    seed_phrase="Mera dil",
    sentiment="Romantic, yearning",
    rhyme_pattern="AABBA",
    tempo=95,
    energy=0.6,
    stanzas=2,
    lines_per_stanza=5
)
print(lyrics)

Generating Romantic Lyrics...

Generated Romantic Lyrics:
 Stanza_1:
: Romantic, love, passion  : {'A': 'na', 'B': 'ye', 'C': 'di', 'D': 'ra'}  : 90  : 0.4  : -30  : 0.6  : (A)
Tere bina dil aayi hai tumse jeevan tham ke dhaage raati hoon mera (A)
1Ummid se pyar ki kasmein hawaisa naUljhe zindagi ko nahin rehtaaz kaise jaadu sazaan tera)Twoke main woh khushi haiya neeli ho gaya hum mein marnaYeh sone na ... yeh sar pe bandhan na oye sabko paancho chale is tarafon uska raaziya banake maane lage intezar hi raaju baaziyan laana dekhla lo milti haar diMiste hain toh wafa tere rasta tujhi sun liye susceptive-2ever girl's eyesight lies hard on the edge of destiny reality checkerRainy rain yahan shaitaan apni adaaon mehfil haan lagta haath haaye tha pata haqdeer taara bolo bewafa waisi daab duniya ab gham marjaat dhadka hone lagaazaar jab dil ek muskayaariyon ka bole mere phir jaati manzil aa jaaniya aur hum itne nasri haseenaaiyya yaaron ke koi na strengthe jo agar jaaye janliya rang churaya

In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import re

# ========================
# 1. Model & Tokenizer Initialization
# ========================
tokenizer = AutoTokenizer.from_pretrained("/kaggle/working/fine_tuned_impyadav_GPT2_hindi_lyrics")
model = AutoModelForCausalLM.from_pretrained("/kaggle/working/fine_tuned_impyadav_GPT2_hindi_lyrics")

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Add special tokens if they don't already exist
special_tokens = [
    "[Sentiment]", "[Rhyme]", "[Tempo]", "[Energy]", "[Loudness]",
    "[Danceability]", "[Speechiness]", "[Acousticness]", "[Instrumentalness]",
    "[Liveness]", "[Valence]", "[Explicit]", "[Popularity]", "[Chroma]",
    "[SpectralContrast]", "[ZeroCrossings]", "[RhymeScheme]"
]
# Only add tokens that don't already exist
tokens_to_add = []
for token in special_tokens:
    if token not in tokenizer.get_vocab():
        tokens_to_add.append(token)

if tokens_to_add:
    tokenizer.add_special_tokens({'additional_special_tokens': tokens_to_add})
    model.resize_token_embeddings(len(tokenizer))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# ========================
# 2. Enhanced Lyrics Generation
# ========================

def generate_lyrics(
    seed, 
    sentiment, 
    rhyme_dict=None, 
    rhyme_scheme=None,
    tempo=120, 
    energy=0.5,
    loudness=-30,
    danceability=0.5,
    max_length=512
):
    model.eval()
    
    # Default rhyme scheme and dict if not provided
    if not rhyme_scheme:
        rhyme_scheme = "['A', 'A', 'B', 'B', 'C', 'C', 'D', 'D']"
    
    if not rhyme_dict:
        rhyme_dict = "{'A': 'na', 'B': 'ye', 'C': 'ra', 'D': 'di'}"
    
    # Build prompt with all required conditioning
    prompt = (
        f"[Sentiment]: {sentiment} {tokenizer.eos_token} "
        f"[Rhyme]: {rhyme_dict} {tokenizer.eos_token} "
        f"[Tempo]: {tempo} {tokenizer.eos_token} "
        f"[Energy]: {energy} {tokenizer.eos_token} "
        f"[Loudness]: {loudness} {tokenizer.eos_token} "
        f"[Danceability]: {danceability} {tokenizer.eos_token} "
        f"[RhymeScheme]: {rhyme_scheme}\n\n"
    )
    
    # Add the seed text to start generation
    if seed:
        prompt += seed
    
    # Tokenize the prompt
    inputs = tokenizer(prompt, return_tensors="pt", padding=True, truncation=True).to(device)
    
    # Generate lyrics with careful parameter tuning
    with torch.no_grad():
        generated_ids = model.generate(
            inputs.input_ids,
            attention_mask=inputs.attention_mask,
            do_sample=True,
            temperature=0.85,  # Good balance between creativity and coherence
            top_p=0.92,
            top_k=50,
            repetition_penalty=1.15,
            max_length=max_length,  # Use only max_length, not max_new_tokens
            pad_token_id=tokenizer.pad_token_id,
            num_return_sequences=1,
            no_repeat_ngram_size=2
        )
    
    # Decode the generated text
    generated_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
    
    # Clean up the generated text (remove prompt parts)
    output = clean_generated_text(generated_text, prompt)
    
    # Format with stanzas and rhyme scheme
    formatted_output = format_lyrics_with_rhyme_scheme(output, rhyme_scheme)
    
    return formatted_output

def clean_generated_text(generated_text, prompt):
    """Remove the prompt part from the generated text and clean up"""
    # Remove the prompt from the beginning
    if generated_text.startswith(prompt):
        output = generated_text[len(prompt):]
    else:
        # If exact match not found, try to find where the generation actually starts
        # by looking for the seed or the first reasonable sentence
        lines = generated_text.split('\n')
        for i, line in enumerate(lines):
            if line and not any(token in line for token in ["[Sentiment]", "[Rhyme]", "[Tempo]"]):
                output = '\n'.join(lines[i:])
                break
        else:
            output = generated_text
    
    # Remove any trailing special tokens or formatting
    output = re.sub(r'\[.*?\]', '', output).strip()
    
    return output

def format_lyrics_with_rhyme_scheme(lyrics_text, rhyme_scheme_str):
    """Format lyrics with stanza structure and rhyme labels"""
    # Parse rhyme scheme
    try:
        rhyme_scheme = eval(rhyme_scheme_str)
        if isinstance(rhyme_scheme, list):
            rhyme_pattern = rhyme_scheme
        else:
            # Default pattern if parsing fails
            rhyme_pattern = ['A', 'A', 'B', 'B', 'C', 'C', 'D', 'D']
    except:
        # Default if parsing completely fails
        rhyme_pattern = ['A', 'A', 'B', 'B', 'C', 'C', 'D', 'D']
    
    # Split lyrics into lines
    lines = [line.strip() for line in lyrics_text.split('\n') if line.strip()]
    
    # Format with stanzas (assume 4 lines per stanza) and add rhyme labels
    formatted_output = ""
    current_stanza = 1
    
    for i, line in enumerate(lines):
        # Start a new stanza
        if i % 4 == 0:
            if i > 0:
                formatted_output += "\n"  # Add blank line between stanzas
            formatted_output += f"Stanza_{current_stanza}:\n"
            current_stanza += 1
        
        # Add line with rhyme label
        rhyme_label = rhyme_pattern[i % len(rhyme_pattern)]
        formatted_output += f"{line} ({rhyme_label})\n"
    
    return formatted_output

# ========================
# 3. Examples with Real Model Generation
# ========================

# Example 1: Romantic song
print("Generating Romantic Lyrics...")
romantic_lyrics = generate_lyrics(
    seed="Tere bina", 
    sentiment="Romantic, love, passion",
    rhyme_dict="{'A': 'na', 'B': 'ye', 'C': 'di', 'D': 'ra'}",
    rhyme_scheme="['A', 'A', 'B', 'B']",
    tempo=90,
    energy=0.4,
    danceability=0.6
)
print("\nGenerated Romantic Lyrics:\n", romantic_lyrics)

# Example 2: Sad song
print("\nGenerating Sad Lyrics...")
sad_lyrics = generate_lyrics(
    seed="Dil ke dard", 
    sentiment="Sad, melancholy, heartbreak",
    rhyme_dict="{'A': 'ra', 'B': 'na', 'C': 'ya', 'D': 'ni'}",
    rhyme_scheme="['A', 'B', 'A', 'B']",
    tempo=70,
    energy=0.3,
    danceability=0.4
)
print("\nGenerated Sad Lyrics:\n", sad_lyrics)

# Example 3: Party song
print("\nGenerating Party Lyrics...")
party_lyrics = generate_lyrics(
    seed="Aaj raat ka", 
    sentiment="Happy, energetic, celebration",
    rhyme_dict="{'A': 'na', 'B': 'le', 'C': 'jo', 'D': 'ra'}",
    rhyme_scheme="['A', 'A', 'A', 'A']",
    tempo=130,
    energy=0.8,
    danceability=0.9
)
print("\nGenerated Party Lyrics:\n", party_lyrics)

# ========================
# 4. Production-Ready Function with Error Handling
# ========================

def generate_hindi_lyrics(
    seed_phrase,
    sentiment="Neutral",
    rhyme_pattern="AABB",  # Options: AABB, ABAB, AAAA, etc.
    tempo=100,
    energy=0.5,
    stanzas=2,
    lines_per_stanza=4
):
    """
    Production-ready function that handles all the complexity and provides a clean interface
    """
    try:
        # Convert rhyme pattern to rhyme scheme list format
        rhyme_scheme_list = []
        for _ in range(stanzas):
            for char in rhyme_pattern[:lines_per_stanza]:
                rhyme_scheme_list.append(char)
        
        rhyme_scheme_str = str(rhyme_scheme_list)
        
        # Common Hindi rhyming sounds based on pattern
        rhyme_dict = {
            'A': 'na',  # ends with 'na' sound
            'B': 'ye',  # ends with 'ye' sound
            'C': 'ra',  # ends with 'ra' sound
            'D': 'di',  # ends with 'di' sound
            'E': 'le',  # ends with 'le' sound
        }
        
        # Generate lyrics using the model
        lyrics = generate_lyrics(
            seed=seed_phrase,
            sentiment=sentiment,
            rhyme_dict=str(rhyme_dict),
            rhyme_scheme=rhyme_scheme_str,
            tempo=tempo,
            energy=energy
        )
        
        return lyrics
    
    except Exception as e:
        print(f"Error during lyric generation: {e}")
        return None

# Example with user input
lyrics = generate_hindi_lyrics("Dil ki baatein", sentiment="Romantic", rhyme_pattern="AABB", tempo=90, energy=0.5)
print("\nGenerated Lyrics:\n", lyrics)


Generating Romantic Lyrics...

Generated Romantic Lyrics:
 Stanza_1:
: Romantic, love, passion  : {'A': 'na', 'B': 'ye', 'C': 'di', 'D': 'ra'}  : 90  : 0.4  : -30  : 0.6  : (A)
Tere bina tum na (A)
1Soche hum hai joh dil mein hua hi tumhare ho naRap se yeh koi tujhe meri deewanaJimmy aasmana chura lo teri saza raGalat ki aankhen ka raaton ko rehnaRober toh churake raat ke paaye gaayega ... yaar nahi jaise kya saraam ne daastan loon maine laad gaya) (2xMachalte hain machal ke taaron mujhko khushi shuruya dhadkan looon do raasta bechain banaya lagta raha zamaana Lagta rooth badha hawa todke duniya batao le gayen pahuncha tere pehchan lagiyan baaki di-3 Franchi galegi chaand sitare apni haath milega phir woh maang layega thoda si dekho jeet ban jaati haaye jabse pyar sabki ka dard barbaadGaye angrejiya yeh hoti haan! Ferrari iska achche bas ungliyon meine saala mere haseeno ke hi dekhun jaaye bolshan latnaja pagalate hi rang jaatarach jaisa bulaya akhiyan thaabujaari parbat jaathi muskaru

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Load model
tokenizer = AutoTokenizer.from_pretrained("impyadav/GPT2-FineTuned-Hinglish-Song-Generation")
model = AutoModelForCausalLM.from_pretrained("impyadav/GPT2-FineTuned-Hinglish-Song-Generation").to("cuda" if torch.cuda.is_available() else "cpu")

def generate_rom_hindi_lyrics(start_phrase):
    """Generate 2 stanzas of Romanized Hindi lyrics with strict formatting"""
    PROMPT = f"""Write two stanzas (8 lines total) of Romanized Hindi lyrics with:
1. Strict AABB rhyme scheme
2. Romance theme
3. Only Romanized Hindi text (no English)
4. Each stanza must have 4 lines
5. Must not repeat phrases
6. Must complete within 8 lines

Example:
Tere bina jeena mushkil sa lage
Dil mein basa ek ajeeb sa dard uthhe
Raat din teri yaad satati hai
Khwabon mein bas teri surat dikhe

Now generate NEW lyrics starting with: "{start_phrase}"
Lyrics:
"""
    inputs = tokenizer(PROMPT, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            inputs.input_ids,
            max_new_tokens=150,  # Reduced length
            temperature=0.8,     # Slightly higher for diversity
            top_k=40,           # Limit vocabulary selection
            top_p=0.9,          # Nucleus sampling
            repetition_penalty=1.5,  # Strong anti-repetition
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    
    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    lyrics = full_text[len(PROMPT):]
    
    # Aggressive cleaning
    lines = []
    for line in lyrics.split('\n'):
        line = line.strip()
        # Filter out lines with repetition or English
        if (line and len(line.split()) > 2 
            and not any(c.isupper() for c in line[:3])
            and not any(word in line for word in ["repeat", "example", "lyrics"])
            and len(set(line.split())) > 3):  # Minimum unique words
            lines.append(line)
    
    # Take first 8 valid lines and format
    lines = lines[:8]
    if len(lines) >= 4:
        stanza1 = '\n'.join(lines[:4])
        stanza2 = '\n'.join(lines[4:8]) if len(lines) > 4 else ""
        return f"{stanza1}\n\n{stanza2}" if stanza2 else stanza1
    return "Could not generate proper lyrics. Please try a different starting phrase."

# Test with different starters
phrases = ["Pyaar hai yeh", "Dil ki dhadkan", "Tere bina", "Raat ki baatein"]
for phrase in phrases:
    print(f"\nInput: '{phrase}'\n{'-'*30}")
    print("Generated Lyrics:\n", generate_rom_hindi_lyrics(phrase))


Input: 'Pyaar hai yeh'
------------------------------
Generated Lyrics:
 Could not generate proper lyrics. Please try a different starting phrase.

Input: 'Dil ki dhadkan'
------------------------------
Generated Lyrics:
 Could not generate proper lyrics. Please try a different starting phrase.

Input: 'Tere bina'
------------------------------
